In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")

In [0]:
import json

from pyspark.sql import functions as F
from common_utils.logging import get_logger
from common_utils.goldfunctions import date_key, write_gold

# ============================================================
# 1. LOGGER
# ============================================================
logger = get_logger("s2g_dimdate")

# ============================================================
# 2. CONFIG PATH
# ============================================================
dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")

logger.info("Reading Gold configuration from %s", config_path)

# ============================================================
# 3. READ CONFIG
# ============================================================
with open(config_path, "r") as file:
    config = json.load(file)

# ============================================================
# 4. CONFIG SECTIONS
# ============================================================
table_config = config["dim_date"]

date_sources = table_config["date_sources"]
target_table = table_config["target_table"]

# ============================================================
# 5. FIND THE DATE RANGE ACROSS ALL FACTS
# ============================================================
all_dates = None

for source in date_sources:
    dates = spark.table(source["table"]).select(F.to_date(source["column"]).alias("date"))
    all_dates = dates if all_dates is None else all_dates.union(dates)

min_date, max_date = all_dates.agg(F.min("date"), F.max("date")).first()

logger.info("Date range: %s to %s", min_date, max_date)

# ============================================================
# 6. BUILD DIM_DATE
# ============================================================
dim_date = (
    spark.sql(f"""
        SELECT explode(sequence(to_date('{min_date}'), to_date('{max_date}'), interval 1 day)) AS date
    """)
    .withColumn("date_key",   date_key("date"))
    .withColumn("year",       F.year("date"))
    .withColumn("quarter",    F.quarter("date"))
    .withColumn("month",      F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("week_start", F.date_trunc("week", "date").cast("date"))
    .withColumn("day_name",   F.date_format("date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("date").isin(1, 7))
)

# ============================================================
# 7. DISPLAY TRANSFORMED DATA
# ============================================================
display(dim_date)

# ============================================================
# 8. WRITE GOLD TABLE
# ============================================================
write_gold(spark, dim_date, target_table)

# ============================================================
# 9. COMPLETED
# ============================================================
logger.info("Gold dim_date processing completed")